# **Title: Week 10 : Document Classification**


**Submitted by:** Banu Boopalan

**Date:** 04/19/25

**Course:** Data Science – DATA620

**Video Link:** 

**Github Repository:

**PERSUADE 2.0 Corpus dataset**

PERSUADE 2.0
This dataset was created by The Learning Agency and Vanderbilt University, originally pulled from here: https://github.com/scrosseye/persuade_corpus_2.0. More info from TLA about the dataset is available here.

url: https://www.kaggle.com/datasets/nbroad/persaude-corpus-2/data
Citation : Bellow Description from Kaggle URL above

"The PERSUADE 2.0 corpus builds on the PERSUADE 1.0 corpus by providing holistic essay scores to each persuasive essay in the PERSUADE 1.0 corpus as well as proficiency scores for each argumentative and discourse element found in the initial corpus. This version also contains all essays (as compared to 1.0 which linked the training set for the Kaggle competition)

In total, the PERSUADE 2.0 corpus comprises over 25,000 argumentative essays produced by 6th-12th grade students in the United States for 15 prompts on two writing tasks: independent and source-based writing. The PERSUADE 2.0 corpus provides detailed individual and demographic information for each writer as well as the initial annotations for argumentative and discourse element found PERSUADE 1.0.
License
(see here) CC BY-NC-SA 4.0 DEED Attribution-NonCommercial-ShareAlike 4.0 International license (https://creativecommons.org/licenses/by-nc-sa/4.0/deed.en)"






**Read the Dataset**

**Import Libraries and Load Dataset**


In [48]:
import pandas as pd

# Read the persuade_2.0.csv file into a DataFrame
file_path = r'C:\Users\Banu\Documents\GitHub\CUNY-DATA-620-WebAnalytics\Week 10\persuade_2.0_human_scores_demo_id_github.csv'  # Update this with the actual path to your CSV file
df = pd.read_csv(file_path)

df.head()



,essay_id_comp,full_text,holistic_essay_score,word_count,prompt_name,task,assignment,source_text,gender,grade_level,ell_status,race_ethnicity,economically_disadvantaged,student_disability_status
0,423A1CA112E2,Phones\n\nModern humans today are always on th...,3,378,Phones and driving,Independent,Today the majority of humans own and operate c...,NaN,M,NaN,NaN,Black/African American,NaN,NaN
1,BC75783F96E3,This essay will explain if drivers should or s...,4,432,Phones and driving,Independent,Today the majority of humans own and operate c...,NaN,M,NaN,NaN,Black/African American,NaN,NaN
2,74C8BC7417DE,Driving while the use of cellular devices\n\nT...,2,179,Phones and driving,Independent,Today the majority of humans own and operate c...,NaN,F,NaN,NaN,White,NaN,NaN
3,A8445CABFECE,Phones & Driving\n\nDrivers should not be able...,3,221,Phones and driving,Independent,Today the majority of humans own and operate c...,NaN,M,NaN,NaN,Black/African American,NaN,NaN
4,6B4F7A0165B9,Cell Phone Operation While Driving\n\nThe abil...,4,334,Phones and driving,Independent,Today the majority of humans own and operate c...,NaN,M,NaN,NaN,White,NaN,NaN


In [49]:


race_counts = df['race_ethnicity'].value_counts().reset_index()
race_counts.columns = ['Race/Ethnicity', 'Count']
print(race_counts)

                   Race/Ethnicity  Count
0                           White  11571
1                 Hispanic/Latino   6560
2          Black/African American   4959
3          Asian/Pacific Islander   1743
4         Two or more races/Other   1022
5  American Indian/Alaskan Native    141


In [50]:

gender_counts = df['gender'].value_counts().reset_index()
gender_counts.columns = ['Gender', 'Count']
print(gender_counts)

  Gender  Count
0      F  13142
1      M  12854


**Preprocess Text converting it to lowercase, removing punctuation, and removing stopwords.**

In [51]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
import nltk
from nltk.corpus import stopwords
import string

nltk.download('stopwords')
def preprocess_text(text):
    stop_words = set(stopwords.words('english'))
    text = text.lower()  
    text = ''.join([char for char in text if char not in string.punctuation])  
    words = text.split()
    words = [word for word in words if word not in stop_words]  
    return ' '.join(words)

df['processed_text'] = df['full_text'].apply(preprocess_text)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Banu\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


**Convert into numerical features using TF-IDF and encode the target variable (gender).**

In [52]:

if 'processed_text' not in df.columns:
    raise ValueError("column is missing")

tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['processed_text']).toarray()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['gender'])

gender_mapping = {index: label for index, label in enumerate(label_encoder.classes_)}
print('Gender Mapping:', gender_mapping)

Gender Mapping: {0: 'F', 1: 'M'}


In [53]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [54]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

**Random Forest Classifier**


In [55]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
target_names = [gender_mapping[0], gender_mapping[1]]



**Logistic Regression Model**


In [56]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)



**(KNN) Model**

In [57]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)
y_pred_knn = knn_model.predict(X_test_scaled)


In [58]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

models = ['Random Forest', 'Logistic Regression', 'KNN']
accuracies = [
    accuracy_score(y_test, y_pred),
    accuracy_score(y_test, y_pred_lr),
    accuracy_score(y_test, y_pred_knn)
]
precisions = [
    precision_score(y_test, y_pred, average='weighted'),
    precision_score(y_test, y_pred_lr, average='weighted'),
    precision_score(y_test, y_pred_knn, average='weighted')
]
recalls = [
    recall_score(y_test, y_pred, average='weighted'),
    recall_score(y_test, y_pred_lr, average='weighted'),
    recall_score(y_test, y_pred_knn, average='weighted')
]
f1_scores = [
    f1_score(y_test, y_pred, average='weighted'),
    f1_score(y_test, y_pred_lr, average='weighted'),
    f1_score(y_test, y_pred_knn, average='weighted')
]
comparison_df = pd.DataFrame({
    'Model': models,
    'Accuracy': accuracies,
    'Precision': precisions,
    'Recall': recalls,
    'F1-Score': f1_scores
})


**Document Classification Using NLTK, use code from book.**
**Here we can see that if the word "appeared" is present, then the document is 2.1 times more likely to belong to class Female**

We can create a classifier with 2000 most frequent words and then see if the word appears in the documents. We can then use these words to calculate the probability of the class.if our vocabulary contains ['students', 'school', 'sports'] and a document contains only 'students' and 'school', the function would return below. So it is easy to learn patterns like "documents containing 'appeared' are more likely to be written by females" etc..

'contains(students)': True,
'contains(school)': True,
'contains(sports)': False




In [ ]:

from nltk import FreqDist
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
import random

documents = [(text.split(), label) for text, label in zip(df['processed_text'], df['gender'])]
random.shuffle(documents)
all_words = FreqDist(word.lower() for text, label in documents for word in text if word not in stopwords.words('english'))
word_features = list(all_words)[:2000]

def document_features(document):
    document_words = set(document)
    features = {}
    for word in word_features:
        features[f'contains({word})'] = (word in document_words)
    return features

feature_sets = [(document_features(text), label) for text, label in documents]
train_set, test_set = train_test_split(feature_sets, test_size=0.2, random_state=42)

classifier = NaiveBayesClassifier.train(train_set)
classifier.show_most_informative_features(25)

 

**Formulas**
Class precision = TP / (TP + FP)
Class recall = TP / (TP + FN)
Class F1-score = 2 * (precision * recall) / (precision + recall)
Class accuracy = (TP + TN) / (TP + TN + FP + FN)


In [ ]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix


summary_df = pd.DataFrame(columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 
                                   'True Positives (TP)', 'True Negatives (TN)', 
                                   'False Positives (FP)', 'False Negatives (FN)'])


models_data = []

# Random Forest
cm_rf = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn_rf, fp_rf, fn_rf, tp_rf = cm_rf.ravel()
models_data.append({
    'Model': 'Random Forest',
    'Accuracy': accuracies[0],
    'Precision': precisions[0],
    'Recall': recalls[0],
    'F1-Score': f1_scores[0],
    'True Positives (TP)': int(tp_rf),
    'True Negatives (TN)': int(tn_rf),
    'False Positives (FP)': int(fp_rf),
    'False Negatives (FN)': int(fn_rf)
})

# Logistic Regression
cm_lr = confusion_matrix(y_test, y_pred_lr, labels=[0, 1])
tn_lr, fp_lr, fn_lr, tp_lr = cm_lr.ravel()
models_data.append({
    'Model': 'Logistic Regression',
    'Accuracy': accuracies[1],
    'Precision': precisions[1],
    'Recall': recalls[1],
    'F1-Score': f1_scores[1],
    'True Positives (TP)': int(tp_lr),
    'True Negatives (TN)': int(tn_lr),
    'False Positives (FP)': int(fp_lr),
    'False Negatives (FN)': int(fn_lr)
})

# KNN
cm_knn = confusion_matrix(y_test, y_pred_knn, labels=[0, 1])
tn_knn, fp_knn, fn_knn, tp_knn = cm_knn.ravel()
models_data.append({
    'Model': 'KNN',
    'Accuracy': accuracies[2],
    'Precision': precisions[2],
    'Recall': recalls[2],
    'F1-Score': f1_scores[2],
    'True Positives (TP)': int(tp_knn),
    'True Negatives (TN)': int(tn_knn),
    'False Positives (FP)': int(fp_knn),
    'False Negatives (FN)': int(fn_knn)
})


summary_df = pd.concat([summary_df, pd.DataFrame(models_data)], ignore_index=True)



C:\Users\Banu\AppData\Local\Temp\ipykernel_24440\808701464.py:58: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  summary_df = pd.concat([summary_df, pd.DataFrame(models_data)], ignore_index=True)


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from tabulate import tabulate

y_test_nb = [label for features, label in test_set]
y_pred_nb = [classifier.classify(features) for features, label in test_set]

accuracy_nb = nltk.classify.accuracy(classifier, test_set)
precision_nb = precision_score(y_test_nb, y_pred_nb, average='weighted', zero_division=0)
recall_nb = recall_score(y_test_nb, y_pred_nb, average='weighted', zero_division=0)
f1_nb = f1_score(y_test_nb, y_pred_nb, average='weighted', zero_division=0)

cm_nb = confusion_matrix(y_test_nb, y_pred_nb, labels=['F', 'M'])
tn_nb, fp_nb, fn_nb, tp_nb = cm_nb.ravel()


naive_bayes_metrics = {
    'Model': 'NLTK Naive Bayes',
    'Accuracy': accuracy_nb,
    'Precision': precision_nb,
    'Recall': recall_nb,
    'F1-Score': f1_nb,
    'True Positives (TP)': int(tp_nb),
    'True Negatives (TN)': int(tn_nb),
    'False Positives (FP)': int(fp_nb),
    'False Negatives (FN)': int(fn_nb)
}

summary_df = pd.concat([summary_df, pd.DataFrame([naive_bayes_metrics])], ignore_index=True)


for col in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    summary_df[col] = summary_df[col].map(lambda x: f'{x:.4f}')

print("\nComprehensive Comparison of All Models:")
print(tabulate(summary_df, headers='keys', tablefmt='grid'))


Comprehensive Comparison of All Models:
+----+---------------------+------------+-------------+----------+------------+-----------------------+-----------------------+------------------------+------------------------+
|    | Model               |   Accuracy |   Precision |   Recall |   F1-Score |   True Positives (TP) |   True Negatives (TN) |   False Positives (FP) |   False Negatives (FN) |
+====+=====================+============+=============+==========+============+=======================+=======================+========================+========================+
|  0 | Random Forest       |     0.6281 |      0.6318 |   0.6281 |     0.6257 |                  1428 |                  1838 |                    754 |                   1180 |
+----+---------------------+------------+-------------+----------+------------+-----------------------+-----------------------+------------------------+------------------------+
|  1 | Logistic Regression |     0.6704 |      0.6706 |   0.6704 |   

**Create Naive Bayes Model for Race/Ethnicity Classification using 2000 most frequent words used function similar to the book**

In [ ]:
from nltk import FreqDist
from nltk.classify import NaiveBayesClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


all_words_race = FreqDist(word.lower() for text in df['processed_text'] for word in text.split())
word_features = list(all_words_race)[:2000]
documents_race = [(text.split(), label) for text, label in zip(df['processed_text'], df['race_ethnicity'])]
def document_features_race(document):
    document_words = set(document)
    features = {}
    for word in word_features:
        features[f'contains({word})'] = (word in document_words)
    return features


feature_sets_race = [(document_features_race(text), label) for text, label in documents_race]
train_set_race, test_set_race = train_test_split(feature_sets_race, test_size=0.2, random_state=42)
classifier_race = NaiveBayesClassifier.train(train_set_race)
accuracy_race = nltk.classify.accuracy(classifier_race, test_set_race)
print(f'Naive Bayes Classifier Accuracy for Race/Ethnicity: {accuracy_race * 100:.2f}%')

y_test_race = [label for features, label in test_set_race]
y_pred_race = [classifier_race.classify(features) for features, label in test_set_race]
classifier_race.show_most_informative_features(10)

Naive Bayes Classifier Accuracy for Race/Ethnicity: 30.83%
Most Informative Features
      contains(diffrent) = True           Americ : Asian/ =     44.5 : 1.0
          contains(fake) = True           Americ : Two or =     26.7 : 1.0
        contains(stupid) = True           Americ : Hispan =     13.3 : 1.0
          contains(boat) = True           Americ : Asian/ =     12.1 : 1.0
        contains(cattle) = True           Americ : Asian/ =     12.1 : 1.0
         contains(wreck) = True           Two or : Asian/ =     11.0 : 1.0
        contains(belive) = True           Americ : Asian/ =      8.9 : 1.0
        contains(killed) = True           Black/ : Asian/ =      8.8 : 1.0
      contains(teamwork) = True           Asian/ : Hispan =      8.7 : 1.0
      contains(atlantic) = True           Americ : Asian/ =      8.7 : 1.0
Most Informative Features
      contains(diffrent) = True           Americ : Asian/ =     44.5 : 1.0
          contains(fake) = True           Americ : Two or =     

**Race-Based Classification Metrics**


In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
import pandas as pd
from tabulate import tabulate

# Get the actual class names from the test set
y_test_race_actual = [label for features, label in test_set_race]
y_pred_race_actual = [classifier_race.classify(features) for features, label in test_set_race]

# Get unique labels in the correct order
unique_labels = sorted(list(set(y_test_race_actual)))

# Calculate overall accuracy and metrics
overall_accuracy = nltk.classify.accuracy(classifier_race, test_set_race)
overall_precision = precision_score(y_test_race_actual, y_pred_race_actual, average='weighted', zero_division=0)
overall_recall = recall_score(y_test_race_actual, y_pred_race_actual, average='weighted', zero_division=0)
overall_f1 = f1_score(y_test_race_actual, y_pred_race_actual, average='weighted', zero_division=0)

# Print overall metrics as text summary
print(f'Race/Ethnicity Classification Performance:')
print(f'Overall Accuracy: {overall_accuracy:.4f}')
print(f'Overall Precision: {overall_precision:.4f}')
print(f'Overall Recall: {overall_recall:.4f}')
print(f'Overall F1-Score: {overall_f1:.4f}\n')

# Calculate and display per-class metrics
print(f'Race/Ethnicity Classification - Per-Class Metrics:')
cm = confusion_matrix(y_test_race_actual, y_pred_race_actual, labels=unique_labels)
class_metrics = []

for i, label in enumerate(unique_labels):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    tn = cm.sum() - (tp + fp + fn)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    
    class_metrics.append({
        'Class': label,
        'True Positives (TP)': int(tp),
        'True Negatives (TN)': int(tn),
        'False Positives (FP)': int(fp),
        'False Negatives (FN)': int(fn),
        'Accuracy': float(accuracy),
        'Precision': float(precision),
        'Recall': float(recall),
        'F1-Score': float(f1)
    })

summary_race = pd.DataFrame(class_metrics)
print(tabulate(summary_race, headers='keys', tablefmt='grid'))

Race/Ethnicity Classification Performance:
Overall Accuracy: 0.3083
Overall Precision: 0.4249
Overall Recall: 0.3083
Overall F1-Score: 0.3302

Race/Ethnicity Classification - Per-Class Metrics:
+----+--------------------------------+-----------------------+-----------------------+------------------------+------------------------+------------+-------------+-----------+------------+
|    | Class                          |   True Positives (TP) |   True Negatives (TN) |   False Positives (FP) |   False Negatives (FN) |   Accuracy |   Precision |    Recall |   F1-Score |
+====+================================+=======================+=======================+========================+========================+============+=============+===========+============+
|  0 | American Indian/Alaskan Native |                     2 |                  4710 |                    462 |                     26 |   0.906154 |  0.00431034 | 0.0714286 | 0.00813008 |
+----+--------------------------------+-------

**Performance for NB model increased from 30.8% to 42.81% with SMOTE (we can balance the minority class by oversampling)**

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

tfidf_race = TfidfVectorizer(max_features=5000)
X_race = tfidf_race.fit_transform(df['processed_text']).toarray()
label_encoder_race = LabelEncoder()
y_race = label_encoder_race.fit_transform(df['race_ethnicity'])
smote = SMOTE(random_state=42)
X_race_smote, y_race_smote = smote.fit_resample(X_race, y_race)
X_train_race, X_test_race, y_train_race, y_test_race = train_test_split(X_race_smote, y_race_smote, test_size=0.2, random_state=42)
nb_model_race_smote = MultinomialNB()
nb_model_race_smote.fit(X_train_race, y_train_race)
y_pred_race_smote = nb_model_race_smote.predict(X_test_race)

accuracy_race_smote = accuracy_score(y_test_race, y_pred_race_smote)
print(f'Naive Bayes Accuracy with SMOTE for Race/Ethnicity Classification: {accuracy_race_smote * 100:.2f}%')
print('Classification Report with SMOTE:\n', classification_report(y_test_race, y_pred_race_smote, target_names=label_encoder_race.classes_))

Naive Bayes Accuracy with SMOTE for Race/Ethnicity Classification: 42.81%
Classification Report with SMOTE:
                                 precision    recall  f1-score   support

American Indian/Alaskan Native       0.74      0.84      0.78      2304
        Asian/Pacific Islander       0.34      0.61      0.44      2316
        Black/African American       0.35      0.25      0.29      2253
               Hispanic/Latino       0.32      0.34      0.33      2318
       Two or more races/Other       0.52      0.25      0.34      2374
                         White       0.35      0.28      0.31      2321

                      accuracy                           0.43     13886
                     macro avg       0.44      0.43      0.42     13886
                  weighted avg       0.44      0.43      0.42     13886

Classification Report with SMOTE:
                                 precision    recall  f1-score   support

American Indian/Alaskan Native       0.74      0.84      0.7


### MultinomialNB Model Results:

In [ ]:
from sklearn.metrics import confusion_matrix

# Create confusion matrix for each class
cm = confusion_matrix(y_test_race, y_pred_race_smote)
class_labels = label_encoder_race.classes_

# Overall metrics (same as before)
accuracy_race_smote = accuracy_score(y_test_race, y_pred_race_smote)
precision_overall = precision_score(y_test_race, y_pred_race_smote, average='weighted', zero_division=0)
recall_overall = recall_score(y_test_race, y_pred_race_smote, average='weighted', zero_division=0)
f1_overall = f1_score(y_test_race, y_pred_race_smote, average='weighted', zero_division=0)

print(f'Overall Performance Metrics:')
print(f'Accuracy: {accuracy_race_smote:.4f}')
print(f'Precision: {precision_overall:.4f}')
print(f'Recall: {recall_overall:.4f}')
print(f'F1-Score: {f1_overall:.4f}')
print('\n')

class_metrics = []
for i, label in enumerate(class_labels):
    true_pos = cm[i, i]
    false_pos = cm[:, i].sum() - true_pos
    false_neg = cm[i, :].sum() - true_pos
    true_neg = cm.sum() - (true_pos + false_pos + false_neg)
    class_precision = true_pos / (true_pos + false_pos) if (true_pos + false_pos) > 0 else 0
    class_recall = true_pos / (true_pos + false_neg) if (true_pos + false_neg) > 0 else 0
    class_f1 = 2 * (class_precision * class_recall) / (class_precision + class_recall) if (class_precision + class_recall) > 0 else 0
    
    class_accuracy = (true_pos + true_neg) / (true_pos + true_neg + false_pos + false_neg)
    class_metrics.append({
        'Class': label,
        'True Positives (TP)': int(true_pos),
        'True Negatives (TN)': int(true_neg),
        'False Positives (FP)': int(false_pos),
        'False Negatives (FN)': int(false_neg),
        'Accuracy': float(class_accuracy),
        'Precision': float(class_precision),
        'Recall': float(class_recall),
        'F1-Score': float(class_f1)
    })

summary_race_smote = pd.DataFrame(class_metrics)
print(tabulate(summary_race_smote, headers='keys', tablefmt='grid'))

Overall Performance Metrics:
Accuracy: 0.4281
Precision: 0.4359
Recall: 0.4281
F1-Score: 0.4154


+----+--------------------------------+-----------------------+-----------------------+------------------------+------------------------+------------+-------------+----------+------------+
|    | Class                          |   True Positives (TP) |   True Negatives (TN) |   False Positives (FP) |   False Negatives (FN) |   Accuracy |   Precision |   Recall |   F1-Score |
+====+================================+=======================+=======================+========================+========================+============+=============+==========+============+
|  0 | American Indian/Alaskan Native |                  1926 |                 10899 |                    683 |                    378 |   0.923592 |    0.738214 | 0.835938 |   0.784042 |
+----+--------------------------------+-----------------------+-----------------------+------------------------+------------------------+---------


### Performance Metrics
- **Accuracy**: The proportion of all predictions that were correct or the Overall correctness of the model
- **Precision**: The proportion of positive predictions that were actually correct. For example, when the model predicts a class, how often is it right?
- **Recall (Sensitivity)**: The proportion of actual positives that were correctly identified. How many of the actual class members did the model capture?
- **F1-Score**: The harmonic mean of precision and recall. Balance between precision and recall

### Analysis of Our Model Results
The model achieves 42.8% accuracy, which is a significant improvement over the 30.8% baseline model without SMOTE, but still not high enough for many production scenarios.
**Class-Specific Performance**:
   - **American Indian/Alaskan Native**: Class has higher performance overall. (Precision: 73.8%, Recall: 83.6%, F1: 78.4%)
   - **Asian/Pacific Islander**: Medium performance for this class (Precision: 34.1%, Recall: 60.9%, F1: 43.7%) 
   - **Black/African American**: Very low recall (25.1%) - the model missed 75% of this group
   - **Hispanic/Latino**: balanced and low metrics (Precision: 31.5%, Recall: 33.7%, F1: 32.6%)
   - **Two or more races/Other**: Precision better (51.9%) but recall is low (25.4%)
   - **White**: In this group both precision and recall is low. (34.5%) and recall (28.2%)

**Imbalance Effects**: SMOTE improved performance for minority classes like American Indian/Alaskan Native.
  Overall accuracy of 42.8% is too low. High variance across classes means concerns around fairness.
  Poor recall across demography is misclassification of data, therefore this model would need to be improved.
  We would have to collect more data, try other methods such as BERT or word embeddings in the future.
